# CSET419 - Introduction to Generative AI
## Lab Assignment: Fabric Scrap Upcycling Concept Generation

**Name:** Srishti Sahi  
**Course:** CSET419 - Introduction to Generative AI

### Objective
Build a Colab-based workflow where a user uploads images of scrap fabrics, the notebook analyzes their visual characteristics, recommends useful upcycled products, and generates AI concept images showing what those fabrics can be turned into.

### Learning Outcomes
- use computer vision to describe uploaded fabric images
- extract dominant colors and patterns from textile scraps
- recommend suitable upcycling products such as pen holders, tote bags, quilts, pouches, and organizers
- generate product concept images from the analyzed fabric description using a diffusion model
- save AI generated results for assignment submission and presentation


## 0. Setup

This notebook is designed for **Google Colab**. A GPU runtime is recommended for faster image generation.


In [ ]:
!pip -q install diffusers transformers accelerate safetensors pillow matplotlib


In [ ]:
import io
import os
import math
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from transformers import BlipForConditionalGeneration, BlipProcessor
from diffusers import AutoPipelineForText2Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


## 1. Upload Scrap Fabric Images

In website demo mode, `Run all` skips the Colab upload widget and waits for images to arrive through the API endpoint instead.


In [ ]:
RUN_INTERACTIVE_NOTEBOOK = False

fabric_images = {}

if RUN_INTERACTIVE_NOTEBOOK:
    from google.colab import files

    uploaded = files.upload()
    for filename, data in uploaded.items():
        image = Image.open(io.BytesIO(data)).convert('RGB')
        fabric_images[filename] = image

    print('Uploaded images:', list(fabric_images.keys()))
else:
    print('Interactive upload skipped. Run the website frontend and send images to the /process API endpoint instead.')


In [ ]:
def show_uploaded_images(images_dict):
    if not images_dict:
        print('No interactive notebook uploads to display.')
        return

    columns = min(3, len(images_dict))
    rows = math.ceil(len(images_dict) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(5 * columns, 4 * rows))
    axes = np.array(axes).reshape(rows, columns)

    for ax in axes.flatten():
        ax.axis('off')

    for ax, (name, image) in zip(axes.flatten(), images_dict.items()):
        ax.imshow(image)
        ax.set_title(name)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_uploaded_images(fabric_images)


## 2. Analyze Fabric Characteristics

The notebook uses a BLIP image captioning model to describe each fabric image and extracts dominant colors to build better product ideas and generation prompts.


In [ ]:
caption_model_name = 'Salesforce/blip-image-captioning-base'
caption_processor = BlipProcessor.from_pretrained(caption_model_name)
caption_model = BlipForConditionalGeneration.from_pretrained(caption_model_name).to(device)

def generate_caption(image):
    inputs = caption_processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        output = caption_model.generate(**inputs, max_new_tokens=30)
    return caption_processor.decode(output[0], skip_special_tokens=True).strip()

def dominant_hex_colors(image, num_colors=5):
    small = image.resize((128, 128))
    palette_image = small.convert('P', palette=Image.ADAPTIVE, colors=num_colors)
    palette = palette_image.getpalette()
    color_counts = sorted(palette_image.getcolors(), reverse=True)

    hex_colors = []
    for count, color_index in color_counts[:num_colors]:
        rgb = palette[color_index * 3: color_index * 3 + 3]
        hex_colors.append('#%02x%02x%02x' % tuple(rgb))
    return hex_colors

def pattern_tags_from_caption(caption):
    text = caption.lower()
    tags = []
    keyword_map = {
        'floral': ['floral', 'flower'],
        'striped': ['stripe', 'striped'],
        'checked': ['check', 'checked', 'plaid'],
        'embroidered': ['embroider', 'embroidery'],
        'denim-like': ['denim', 'jean'],
        'printed': ['print', 'printed', 'pattern'],
        'textured': ['texture', 'woven', 'knit'],
        'patchwork-friendly': ['colorful', 'multi', 'multicolor', 'patterned']
    }

    for tag, keywords in keyword_map.items():
        if any(keyword in text for keyword in keywords):
            tags.append(tag)

    if not tags:
        tags.append('versatile fabric')
    return tags

def analyze_fabric(image):
    caption_result = generate_caption(image)
    colors = dominant_hex_colors(image, num_colors=5)
    tags = pattern_tags_from_caption(caption_result)
    return {
        'caption': caption_result,
        'colors': colors,
        'tags': tags,
    }


In [ ]:
FABRIC_SIZE_HINT = 'mixed'  # choose from: small, medium, large, mixed
TOP_K_PRODUCTS = 4

PRODUCT_CATALOG = [
    {'name': 'pen holder', 'min_size': 'small', 'score': 8, 'reason': 'small scraps can wrap cardboard or stiffened bases well'},
    {'name': 'zip pouch', 'min_size': 'small', 'score': 9, 'reason': 'works well with decorative scraps and lining'},
    {'name': 'bookmark set', 'min_size': 'small', 'score': 7, 'reason': 'uses thin strips and printed scraps efficiently'},
    {'name': 'coaster set', 'min_size': 'small', 'score': 7, 'reason': 'good for repeated small square pieces'},
    {'name': 'tote bag', 'min_size': 'medium', 'score': 10, 'reason': 'best for larger coordinated pieces'},
    {'name': 'laptop sleeve', 'min_size': 'medium', 'score': 8, 'reason': 'structured fabric scraps work well with padding'},
    {'name': 'cushion cover', 'min_size': 'medium', 'score': 9, 'reason': 'printed fabrics show strongly on flat surfaces'},
    {'name': 'storage basket', 'min_size': 'medium', 'score': 8, 'reason': 'thicker or layered scraps can create form'},
    {'name': 'quilt panel', 'min_size': 'large', 'score': 10, 'reason': 'ideal for combining many scraps into one composition'},
    {'name': 'wall hanging', 'min_size': 'large', 'score': 8, 'reason': 'good for expressive textile patterns and color blocks'},
]

SIZE_ORDER = {'small': 1, 'medium': 2, 'large': 3, 'mixed': 3}

def recommend_products(analysis, size_hint='mixed', top_k=4):
    text = (analysis['caption'] + ' ' + ' '.join(analysis['tags'])).lower()
    scored = []

    for item in PRODUCT_CATALOG:
        if SIZE_ORDER[size_hint] < SIZE_ORDER[item['min_size']]:
            continue

        score = item['score']
        if item['name'] in ['quilt panel', 'cushion cover', 'wall hanging'] and ('printed' in text or 'floral' in text or 'patchwork' in text):
            score += 2
        if item['name'] in ['tote bag', 'storage basket', 'laptop sleeve'] and ('denim' in text or 'textured' in text or 'woven' in text):
            score += 2
        if item['name'] in ['pen holder', 'coaster set', 'bookmark set', 'zip pouch'] and size_hint in ['small', 'mixed']:
            score += 1

        scored.append({
            'product': item['name'],
            'score': score,
            'reason': item['reason'],
        })

    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:top_k]

fabric_analysis = {}
for filename, image in fabric_images.items():
    analysis = analyze_fabric(image)
    analysis['recommendations'] = recommend_products(analysis, size_hint=FABRIC_SIZE_HINT, top_k=TOP_K_PRODUCTS)
    fabric_analysis[filename] = analysis

if fabric_analysis:
    fabric_analysis
else:
    print('Interactive notebook analysis skipped. The API will analyze uploaded website images at request time.')


In [ ]:
def visualize_analysis(results):
    if not results:
        print('No interactive notebook analysis results to display.')
        return

    for filename, item in results.items():
        print('=' * 100)
        print('Image:', filename)
        print('Caption:', item['caption'])
        print('Pattern Tags:', ', '.join(item['tags']))
        print('Dominant Colors:', ', '.join(item['colors']))
        print('Recommended Products:')
        for rec in item['recommendations']:
            print(f"  - {rec['product']} (score {rec['score']}): {rec['reason']}")

visualize_analysis(fabric_analysis)


## 3. Load the Diffusion Model

The notebook uses `stabilityai/sd-turbo` to quickly generate concept renders for each recommended product.


In [ ]:
model_id = 'stabilityai/sd-turbo'

pipe_kwargs = {'torch_dtype': torch.float16 if device == 'cuda' else torch.float32}
if device == 'cuda':
    pipe_kwargs['variant'] = 'fp16'

image_pipe = AutoPipelineForText2Image.from_pretrained(model_id, **pipe_kwargs)
image_pipe = image_pipe.to(device)
image_pipe.set_progress_bar_config(disable=True)

print('Image generation pipeline ready.')


## 4. Build Prompts for Upcycled Products

These prompts convert the observed fabric characteristics into product-render instructions for the diffusion model.


In [ ]:
NEGATIVE_PROMPT = 'blurry, distorted, extra handles, extra pockets, text, watermark, low quality, bad anatomy, duplicate objects'

def build_generation_prompt(product_name, analysis):
    color_text = ', '.join(analysis['colors'][:4])
    tag_text = ', '.join(analysis['tags'])
    caption = analysis['caption']

    prompt = (
        f'a realistic handcrafted {product_name} made from upcycled scrap fabric, '
        f'inspired by this textile description: {caption}. '
        f'fabric characteristics: {tag_text}. '
        f'color palette: {color_text}. '
        f'clean studio product photography, sustainable design prototype, detailed stitching, '
        f'high quality, centered composition, soft lighting'
    )
    return prompt

if fabric_analysis:
    for filename, item in fabric_analysis.items():
        print('\n' + '=' * 100)
        print('Prompt previews for:', filename)
        for rec in item['recommendations']:
            prompt = build_generation_prompt(rec['product'], item)
            print(f"\n{rec['product'].title()} Prompt:\n{prompt}")
else:
    print('Interactive notebook prompt preview skipped. Prompts will be built per API request.')


## 5. Generate Concept Images

Edit the variables below if you want fewer or more outputs. One image per recommended product is enough for a lab submission, but you can generate more variants.


In [ ]:
MAX_PRODUCTS_PER_FABRIC = 3
IMAGES_PER_PRODUCT = 1
OUTPUT_DIR = Path('generated_stitch_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

generated_results = []

for filename, item in fabric_analysis.items():
    image_stem = Path(filename).stem.replace(' ', '_')
    selected_products = item['recommendations'][:MAX_PRODUCTS_PER_FABRIC]

    for rec in selected_products:
        product_name = rec['product']
        prompt = build_generation_prompt(product_name, item)

        for variant_idx in range(IMAGES_PER_PRODUCT):
            result = image_pipe(
                prompt=prompt,
                negative_prompt=NEGATIVE_PROMPT,
                num_inference_steps=2,
                guidance_scale=0.0,
                height=512,
                width=512,
                generator=torch.Generator(device=device).manual_seed(SEED + variant_idx),
            ).images[0]

            output_name = f'{image_stem}_{product_name.replace(" ", "_")}_{variant_idx + 1}.png'
            output_path = OUTPUT_DIR / output_name
            result.save(output_path)

            generated_results.append({
                'source_image': filename,
                'product': product_name,
                'prompt': prompt,
                'path': str(output_path),
                'image': result,
            })

if generated_results:
    print('Generated images:', len(generated_results))
else:
    print('Interactive notebook generation skipped. The API will generate outputs when the website sends a request.')


In [ ]:
def show_generated_results(results):
    if not results:
        print('No interactive notebook generated images to display.')
        return

    columns = 3
    rows = math.ceil(len(results) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(5 * columns, 5 * rows))
    axes = np.array(axes).reshape(rows, columns)

    for ax in axes.flatten():
        ax.axis('off')

    for ax, item in zip(axes.flatten(), results):
        ax.imshow(item['image'])
        ax.set_title(f"{item['product']}\nfrom {Path(item['source_image']).stem}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_generated_results(generated_results)


## 6. Save Results for Submission

This cell packages the generated concept images into a zip file so they can be downloaded from Colab.


In [ ]:
archive_path = None

if generated_results:
    archive_path = shutil.make_archive('generated_stitch_outputs', 'zip', root_dir=OUTPUT_DIR)
    print('Saved zip file:', archive_path)
else:
    print('Interactive notebook zip export skipped. The API creates a zip per website request.')


In [ ]:
if archive_path and RUN_INTERACTIVE_NOTEBOOK:
    from google.colab import files
    files.download('generated_stitch_outputs.zip')
else:
    print('Interactive notebook download skipped.')


## 7. Conclusion

This notebook demonstrates an end-to-end generative AI application for textile waste reuse. The workflow accepts scrap fabric images, extracts visual cues, recommends suitable products, and creates AI-generated concept renders of sustainable upcycling outcomes such as tote bags, pen holders, quilt panels, and organizers.

### Suggested Discussion Points for Viva or Report
- why BLIP was used to describe uploaded fabric images
- how dominant colors improve prompt quality for image generation
- how product recommendations depend on fabric size and pattern
- why diffusion models are useful for generating design concepts before physical production
- how this system can help reduce textile waste through reuse planning


## 8. API Mode for Website Demo

These cells expose the same notebook workflow through a temporary HTTP API so the website frontend can call the Colab runtime without showing the notebook UI.


In [ ]:
!pip -q install flask flask-cors


In [ ]:
import base64
import io
import re
import shutil
import subprocess
import threading
import time
import uuid
from pathlib import Path

import torch
from PIL import Image
from flask import Flask, jsonify, request, send_file
from flask_cors import CORS

BASE_DIR = Path('/content/stitcher_api')
RUNS_DIR = BASE_DIR / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def to_data_url(image):
    buffer = io.BytesIO()
    image.save(buffer, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('utf-8')


In [ ]:
app = Flask(__name__)
CORS(app)
api_lock = threading.Lock()

@app.get('/health')
def health():
    return {'status': 'ok', 'device': device}

@app.post('/process')
def process():
    files = request.files.getlist('images')
    if not files:
        return jsonify({'error': 'Upload at least one fabric image.'}), 400

    size_hint = request.form.get('sizeHint', FABRIC_SIZE_HINT)
    top_k = int(request.form.get('topKProducts', TOP_K_PRODUCTS))
    max_products_per_fabric = int(request.form.get('maxProductsPerFabric', MAX_PRODUCTS_PER_FABRIC))
    images_per_product = int(request.form.get('imagesPerProduct', IMAGES_PER_PRODUCT))

    job_id = uuid.uuid4().hex
    job_dir = RUNS_DIR / job_id
    output_dir = job_dir / 'generated_stitch_outputs'
    output_dir.mkdir(parents=True, exist_ok=True)

    fabric_images = {}
    for file in files:
        image = Image.open(io.BytesIO(file.read())).convert('RGB')
        fabric_images[file.filename] = image

    fabric_analysis = {}
    for filename, image in fabric_images.items():
        analysis = analyze_fabric(image)
        analysis['recommendations'] = recommend_products(analysis, size_hint=size_hint, top_k=top_k)
        fabric_analysis[filename] = analysis

    generated_results = []

    with api_lock:
        for filename, item in fabric_analysis.items():
            image_stem = Path(filename).stem.replace(' ', '_')
            selected_products = item['recommendations'][:max_products_per_fabric]

            for rec in selected_products:
                product_name = rec['product']
                prompt = build_generation_prompt(product_name, item)

                for variant_idx in range(images_per_product):
                    result = image_pipe(
                        prompt=prompt,
                        negative_prompt=NEGATIVE_PROMPT,
                        num_inference_steps=2,
                        guidance_scale=0.0,
                        height=512,
                        width=512,
                        generator=torch.Generator(device=device).manual_seed(SEED + variant_idx),
                    ).images[0]

                    output_name = f"{image_stem}_{product_name.replace(' ', '_')}_{variant_idx + 1}.png"
                    output_path = output_dir / output_name
                    result.save(output_path)

                    generated_results.append({
                        'source_image': filename,
                        'product': product_name,
                        'prompt': prompt,
                        'preview': to_data_url(result),
                    })

    archive_path = shutil.make_archive(str(job_dir / 'generated_stitch_outputs'), 'zip', root_dir=output_dir)

    fabrics = []
    for filename, image in fabric_images.items():
        fabrics.append({
            'filename': filename,
            'preview': to_data_url(image),
            'analysis': fabric_analysis[filename],
        })

    return jsonify({
        'jobId': job_id,
        'device': device,
        'downloadUrl': f'/download/{job_id}',
        'fabrics': fabrics,
        'generatedResults': generated_results,
        'meta': {
            'sizeHint': size_hint,
            'topKProducts': top_k,
            'maxProductsPerFabric': max_products_per_fabric,
            'imagesPerProduct': images_per_product,
            'generatedImageCount': len(generated_results),
        },
    })

@app.get('/download/<job_id>')
def download(job_id):
    archive_path = RUNS_DIR / job_id / 'generated_stitch_outputs.zip'
    return send_file(archive_path, as_attachment=True, download_name='generated_stitch_outputs.zip')


In [ ]:
import os

PORT = 5000

# Best run from a fresh runtime, but clean up old tunnel processes when rerunning this cell.
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)

if not os.path.exists('cloudflared'):
    download_result = subprocess.run(
        [
            'wget',
            '-q',
            '-O',
            'cloudflared',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        ],
        check=False,
    )
    if download_result.returncode != 0:
        raise RuntimeError('Could not download cloudflared binary.')
    subprocess.run(['chmod', '+x', 'cloudflared'], check=True)

thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False),
    daemon=True,
)
thread.start()

time.sleep(3)

cloudflared_process = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
start = time.time()

while time.time() - start < 30:
    line = cloudflared_process.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError('Could not get Cloudflare tunnel URL.')

print('Public URL:', public_url)
print('Process endpoint:', f'{public_url}/process')
print('Health endpoint:', f'{public_url}/health')
